# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a structured template for loading, exploring, and processing the dataset via the [`mlcroissant`](https://github.com/mlcommons/croissant) library following Croissant standards.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata is an object with attributes
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, their `@id`s, and field definitions. All entities are referenced by their `@id` according to Croissant best practices.

In [ ]:
# List all record sets in the dataset

print('Available record sets:')
for rs in dataset.record_sets():
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '(no name)')}")

# For demonstration, print fields for the first record set
record_sets = list(dataset.record_sets())
if record_sets:
    first_rs = record_sets[0]
    print(f"\nFields in record set @id={first_rs.id}:")
    for fld in first_rs.fields:
        if hasattr(fld, 'id'):
            print(f"- @id: {fld.id} | name: {getattr(fld, 'name', '(no name)')} | type: {getattr(fld, 'data_type', '')}")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. All lookups, operations, and mappings use record set and field `@id` where applicable.

In [ ]:
# Get the @id for each record set
record_sets = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns from the first record set
if record_sets:
    main_rs_id = record_sets[0]
    print(f"Fields (@id) in first record set {main_rs_id}:")
    print(list(dataframes[main_rs_id].columns))
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps like filtering, normalization, and grouping using fields referenced by their `@id`.

In [ ]:
import numpy as np

# Choose main record set
record_set_id = main_rs_id  # from previous extraction
df = dataframes[record_set_id]

# Find numeric fields by inspecting data types
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric fields available.")

# Example: filter where numeric value > threshold
threshold = df[numeric_field_id].mean() if numeric_fields else None

if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a likely categorical field (e.g., 'gender' or similar id)
    group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    group_field = group_fields[0] if group_fields else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions, for example histograms of a numeric field, boxplots, or grouped bar plots, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for main numeric field
if numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group (if group_field present)
if group_field is not None:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook provided an end-to-end workflow for exploring and analyzing the [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

Key steps:
- Loaded dataset metadata and listed record sets with their Croissant `@id`s.
- Loaded data into pandas DataFrames by record set `@id`.
- Performed basic statistics, filtering and normalization referencing all fields by their `@id`.
- Visualized numeric distributions and grouped summaries to aid further health data analysis.

For deeper analysis, further explore field semantics in the schema and use descriptive field `@id`s directly for subsetting and complex transformations.